In [ ]:
# Import necessary libraries
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

#import one csv
df = pd.read_csv("feature_matrix.csv")

#redo above work with this df, including train test and valid
X = df.drop(columns=['TARGET'])
y = df['TARGET']

#X train test valid
X_train = X[:int(len(X)*0.7)]
X_test = X[int(len(X)*0.7):int(len(X)*0.85)]
X_valid = X[int(len(X)*0.85):]

#y train test valid
y_train = y[:int(len(y)*0.7)]
y_test = y[int(len(y)*0.7):int(len(y)*0.85)]
y_valid = y[int(len(y)*0.85):]

In [ ]:
# Step 1: Standardize the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)

# Step 2: Perform PCA
pca = PCA()
X_pca = pca.fit_transform(X_train)

# Step 3: Analyze Explained Variance
explained_variance = pca.explained_variance_ratio_
cumulative_variance = explained_variance.cumsum()

# Plot cumulative explained variance
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o', linestyle='--')
plt.title('Cumulative Variance Explained by Principal Components')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.grid()
plt.show()

loss_data = {}

# Display the explained variance and dimensionality loss
for i, cumulative in enumerate(cumulative_variance):
    if cumulative not in loss_data:
        loss_data[cumulative] = i+1

In [ ]:
def get_components(explained_var):
    for i in loss_data.keys():
        if i > explained_var:
            return loss_data[i]

In [ ]:
n_components = get_components(1.0)

# Perform PCA with selected number of components
print(n_components)
pca_selected = PCA(n_components=n_components)
X_pca_selected = pca_selected.fit_transform(X_train)
X_pca_test = pca_selected.transform(X_test)
X_pca_valid = pca_selected.transform(X_valid)

In [ ]:
import lightgbm as lgb
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
import seaborn as sns
import matplotlib.pyplot as plt

# Creating and training the LightGBM classifier

'''
Params to search:
1. Information Gain (set max_depth higher to account for deep branches)
2. L2 lambda and L1 alpha (implement elastic net)
    2a. reg param
    2b. l1/l2 ratio
3. scale_pos_weight
'''

clf = lgb.LGBMClassifier(max_depth=31, reg_lambda=0.5, scale_pos_weight=1.4, n_estimators=1000, learning_rate=0.01)
clf.fit(X_pca_selected, y_train, eval_set=(X_pca_valid, y_valid),
        callbacks=[
            lgb.early_stopping(stopping_rounds=100)
            ])



In [ ]:
# Making predictions
cutoff = 0.31
y_pred = (clf.predict_proba(X_pca_test)[:,1] > cutoff).astype(int)
print(y_pred)
# Evaluating the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

contingency_table = pd.crosstab(pd.Series(y_test, name="Actual"), pd.Series(y_pred, name="Predicted"))

# Heatmap for the confusion matrix
sns.heatmap(contingency_table, annot=True, fmt="d", cmap="Blues")
plt.title("Contingency Table")
plt.show()

y_pred_proba = clf.predict_proba(X_pca_test)[:,1]
# print(y_pred_proba[:,0])
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
auc_score = roc_auc_score(y_test, y_pred_proba)

plt.figure()
plt.plot(fpr, tpr, label=f"AUC = {auc_score:.2f}")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Receiver Operating Characteristic (ROC) Curve")
plt.legend(loc="lower right")
plt.show()

print("f1 score: " + str(f1_score(y_test, y_pred)))
print("best iteration:  " + str(clf.best_iteration_))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Assuming you already have your dataset as a DataFrame
# Replace this with your actual data loading process
data = {
    "y_true": y_test,
    "y_pred_proba": y_pred_proba
}
df = pd.DataFrame(data)

# Define the number of bins
num_bins = 20
bins = np.linspace(0, 1, num_bins + 1)

# Separate data based on y_true
probas_0 = df[df['y_true'] == 0]['y_pred_proba']
probas_1 = df[df['y_true'] == 1]['y_pred_proba']

# Create histograms for each y_true value
hist_0, _ = np.histogram(probas_0, bins=bins)
hist_1, _ = np.histogram(probas_1, bins=bins)

# Plot the histogram
bar_width = (bins[1] - bins[0]) * 0.4  # Narrower bars for clarity
bin_centers = (bins[:-1] + bins[1:]) / 2

plt.figure(figsize=(12, 6))
plt.bar(bin_centers - bar_width / 2, hist_0, width=bar_width, label='y_true = 0', alpha=0.7, color='blue')
plt.bar(bin_centers + bar_width / 2, hist_1, width=bar_width, label='y_true = 1', alpha=0.7, color='orange')

# Add labels and legend
plt.xlabel('Predicted Probability')
plt.ylabel('Count')
plt.title('Histogram of Predicted Probabilities by True Label')
plt.legend()
plt.xticks(bins)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
auc_score